In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_iris
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom

# Step 1: Load and Preprocess the Data
iris = load_iris()
X = iris.data # Extracting features
y = iris.target # Labels (not used for training SOM)

# Normalize the data (SOM performs better with scaled data)
scaler = MinMaxScaler()
X_scaled = scaler.fit_transform(X)

# Step 2: Initialize and Train the SOM
som_size = (10, 10) # Grid size (10x10)
som = MiniSom(x=som_size, y=som_size, input_len=X.shape, sigma=1.0, learning_rate=0.5, random_seed=42)

som.random_weights_init(X_scaled)
som.train_random(X_scaled, num_iteration=1000) # Training SOM with 1000 iterations

# Step 3: Compute U-Matrix
u_matrix = np.zeros(som_size) # Initialize U-Matrix

# Function to compute distances between neurons
def neuron_distance(w1, w2):
    return np.linalg.norm(w1 - w2)

# Get the trained weights
weights = som.get_weights()

# Iterate through each neuron in the SOM grid
for i in range(som_size):
    for j in range(som_size):
        # Get the weight vector of the current neuron
        w1 = weights[i, j]

        # Collect distances from neighbors
        distances = []
        for di, dj in [(-1, 0), (1, 0), (0, -1), (0, 1)]: # Left, Right, Up, Down neighbors
            ni, nj = i + di, j + dj
            if 0 <= ni < som_size and 0 <= nj < som_size: # Check bounds
                w2 = weights[ni, nj]
                distances.append(neuron_distance(w1, w2))

        u_matrix[i, j] = np.mean(distances) if distances else 0 # Compute mean distance

# Step 4: Visualize the U-Matrix
plt.figure(figsize=(8, 6))
plt.imshow(u_matrix, cmap="coolwarm", interpolation="nearest")
plt.colorbar(label="Neuron Distance (U-Matrix)")
plt.title("U-Matrix Visualization of SOM on Iris Dataset")
plt.show()

ModuleNotFoundError: No module named 'minisom'

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from minisom import MiniSom
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler

# Load and preprocess the dataset
iris = load_iris()
X = iris.data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Define different SOM configurations
som_configs = [
    {"grid_size": (10, 10), "learning_rate": 0.5, "sigma": 1.0},
    {"grid_size": (15, 15), "learning_rate": 0.3, "sigma": 0.5},
    {"grid_size": (8, 8), "learning_rate": 0.1, "sigma": 2.0},
]

# Function to train SOM and plot U-Matrix
def train_and_plot_som(grid_size, learning_rate, sigma, title):
    som = MiniSom(grid_size, grid_size, X_scaled.shape, sigma=sigma, learning_rate=learning_rate)
    som.random_weights_init(X_scaled)
    som.train_random(X_scaled, 1000) # Train for 1000 iterations

    # Compute U-Matrix
    umatrix = np.zeros(grid_size)
    for i in range(grid_size):
        for j in range(grid_size):
            neighbors = [
                (i + dx, j + dy)
                for dx, dy in [(-1, 0), (1, 0), (0, -1), (0, 1)]
                if 0 <= i + dx < grid_size and 0 <= j + dy < grid_size
            ]
            distances = [np.linalg.norm(som.get_weights()[i, j] - som.get_weights()[n, n]) for n in neighbors]
            umatrix[i, j] = np.mean(distances) if distances else 0

    # Plot U-Matrix
    plt.figure(figsize=(6, 5))
    plt.imshow(umatrix, cmap="coolwarm", interpolation="nearest")
    plt.colorbar(label="Neuron Distance (U-Matrix)")
    plt.title(title)
    plt.show()

# Train and plot SOMs with different configurations
for config in som_configs:
    title = f"SOM Grid: {config['grid_size']}, LR: {config['learning_rate']}, Sigma: {config['sigma']}"
    train_and_plot_som(config['grid_size'], config['learning_rate'], config['sigma'], title)

In [ ]:
import numpy as np
import pandas as pd
from sklearn import datasets
from sklearn.preprocessing import MinMaxScaler
import matplotlib.pyplot as plt
from minisom import MiniSom

# Load Iris Dataset
iris = datasets.load_iris()
data = iris.data
target = iris.target
target_names = iris.target_names

# Normalize the data
scaler = MinMaxScaler()
data = scaler.fit_transform(data)

# Initialize SOM parameters
som_grid_x, som_grid_y = 10, 10 # SOM grid dimensions
som = MiniSom(som_grid_x, som_grid_y, data.shape, sigma=1.0, learning_rate=0.5)
som.random_weights_init(data)
som.train_random(data, 1000)

# # 1. Map data points to their BMUs
bmus = np.array([som.winner(x) for x in data])

# # 2. Visualize clusters with respect to the target variable
plt.figure(figsize=(10, 8))
colors = ['red', 'green', 'blue']
for t, color in zip(range(3), colors):
    plt.scatter(
        bmus[target == t, 0] + np.random.rand(len(bmus[target == t])) * 0.3 - 0.15,
        bmus[target == t, 1] + np.random.rand(len(bmus[target == t])) * 0.3 - 0.15,
        label=target_names[t], color=color, alpha=0.7,
    )
plt.title("Cluster Visualization with SOM")
plt.xlabel("SOM Grid X")
plt.ylabel("SOM Grid Y")
plt.legend()
plt.grid()
plt.show()

# # 3. Analyze SOM separation and compare with ground truth
from sklearn.metrics import adjusted_rand_score, silhouette_score

# Assign cluster labels based on BMUs
unique_bmus = np.unique(bmus, axis=0)
cluster_labels = np.zeros(len(bmus))
for idx, bmu in enumerate(unique_bmus):
    cluster_labels[np.all(bmus == bmu, axis=1)] = idx

# Compute metrics
ari = adjusted_rand_score(target, cluster_labels)
silhouette = silhouette_score(data, cluster_labels)

print("Adjusted Rand Index (ARI):", ari)
print("Silhouette Score:", silhouette)

# Overlay U-Matrix visualization
u_matrix = som.distance_map()
plt.figure(figsize=(10, 8))
plt.pcolor(u_matrix.T, cmap='coolwarm')
plt.colorbar(label='Neuron Distance')
plt.title("U-Matrix of SOM")
plt.xlabel("SOM Grid X")
plt.ylabel("SOM Grid Y")
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom
import seaborn as sns

# Load Titanic dataset
data = sns.load_dataset('titanic')

# Select features for clustering (e.g., age, fare)
data = data[['age', 'fare']].dropna()

# Normalize the data
scaler = MinMaxScaler()
data_scaled = scaler.fit_transform(data)

# Initialize SOM
som_grid_x, som_grid_y = 10, 10 # SOM grid dimensions
som = MiniSom(som_grid_x, som_grid_y, data_scaled.shape, sigma=1.0, learning_rate=0.5)
som.random_weights_init(data_scaled)
som.train_random(data_scaled, 1000)

# Map data points to BMUs
bmus = np.array([som.winner(x) for x in data_scaled])

# Add BMU information to the original data for analysis
data['BMU_x'] = bmus[:, 0]
data['BMU_y'] = bmus[:, 1]

# Visualize U-Matrix
u_matrix = som.distance_map()
plt.figure(figsize=(10, 8))
plt.pcolor(u_matrix.T, cmap='coolwarm')
plt.colorbar(label='Neuron Distance')
plt.title("U-Matrix of SOM")
plt.xlabel("SOM Grid X")
plt.ylabel("SOM Grid Y")
plt.show()

# Visualize clusters on SOM grid
plt.figure(figsize=(10, 8))
plt.scatter(data['BMU_x'], data['BMU_y'], c=data['fare'], cmap='viridis', s=50, alpha=0.7)
plt.colorbar(label='Fare')
plt.title("Passenger Clustering by Fare")
plt.xlabel("BMU X")
plt.ylabel("BMU Y")
plt.grid()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix, classification_report
from minisom import MiniSom

# Load dataset
url = "https://storage.googleapis.com/download.tensorflow.org/data/creditcard.csv"
data = pd.read_csv(url)

# Select relevant features and labels
features = data.drop(['Class', 'Time'], axis=1) # Drop irrelevant columns
labels = data['Class'] # Target column (0 = Normal, 1 = Fraudulent)

# Normalize features
scaler = MinMaxScaler()
features_scaled = scaler.fit_transform(features)

# Initialize SOM
som_grid_x, som_grid_y = 15, 15 # Grid size for SOM
som = MiniSom(som_grid_x, som_grid_y, features_scaled.shape, sigma=1.0, learning_rate=0.5)
som.random_weights_init(features_scaled)
som.train_random(features_scaled, 1000)

# Compute Best Matching Units (BMUs) for each data point
bmus = np.array([som.winner(x) for x in features_scaled])

# Calculate U-Matrix for visualization
u_matrix = som.distance_map()

# Visualize U-Matrix
plt.figure(figsize=(10, 8))
plt.pcolor(u_matrix.T, cmap='coolwarm')
plt.colorbar(label='Neuron Distance (U-Matrix)')
plt.title("U-Matrix (Anomaly Detection)")
plt.xlabel("SOM Grid X")
plt.ylabel("SOM Grid Y")
plt.show()

# Plot BMU density (highlight sparse regions as anomalies)
plt.figure(figsize=(10, 8))
plt.scatter(bmus[:, 0], bmus[:, 1], c=labels, cmap='viridis', s=10, alpha=0.7)
plt.colorbar(label='Fraud (1) vs Normal (0)')
plt.title("BMU Density Map with Labels")
plt.xlabel("BMU X")
plt.ylabel("BMU Y")
plt.grid()
plt.show()

# Identify anomalies based on U-Matrix threshold
threshold = np.percentile(u_matrix, 95) # Top 5% neurons
anomaly_map = (u_matrix > threshold)

# Predict anomalies
predicted_anomalies = np.array([anomaly_map[x, y] for x, y in bmus])
actual_anomalies = labels.values

# Confusion Matrix and Classification Report
print("Confusion Matrix:")
print(confusion_matrix(actual_anomalies, predicted_anomalies))
print("\nClassification Report:")
print(classification_report(actual_anomalies, predicted_anomalies))